In [ ]:
import os
import sys
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import ExperimentConfig
from src.physics.operators import build_true_H0
from src.devices.qutip_device import HiddenDevice
from src.calibration.closed_loop import (
    build_scaled_pulse,
    measure_cz_hardware_cost,
    evaluate_oracle_fidelity,
    run_multi_seed_calibration,
)
from src.calibration.training_loops import make_setup, make_cz_optimizer, train_pulse
from src.experiments.ramsey import reconstruct_pulse_rad

# Load configuration and initialize the "true" hardware fridge
config = ExperimentConfig.load("configs/two_qubit_CZ_zz.yaml")
dt = config.pulse.dt_ns
fridge = HiddenDevice.from_config(config)

# Build JAX Hamiltonian/operators for passive Oracle evaluation
H0_true, Hz_control = build_true_H0(config)

print(
    f"Loaded HiddenDevice | ZZ crosstalk: {config.hardware.zz_crosstalk_mhz} MHz | "
    f"Shots: {config.hardware.n_shots}"
)

In [ ]:
# Arm A: Load the Digital Twin's optimized pulse from NB01
cal_pulse_rad = np.load(config.artifact_path("cal_pulse_rad.npy"))

# Arm B: Smooth pulse designed on the uncalibrated nominal twin (from NB01 initial guess)
# Uses the exact same Fourier/Slepian parameterization, but with uncalibrated parameters
_, _, naive_loss_fn, naive_init, basis, kernel, f_max = make_setup(
    config, n_harmonics=8, coupling_mhz=24.0, detuning_mhz=610.0, anharmonicity_mhz=-300.0
)
naive_steps = 500
naive_opt_params, _ = train_pulse(
    naive_loss_fn, naive_init, make_cz_optimizer(0.01, naive_steps), naive_steps
)
naive_pulse_rad = reconstruct_pulse_rad(naive_opt_params, basis, kernel, f_max)

oracle_a0 = evaluate_oracle_fidelity(cal_pulse_rad, dt, H0_true, Hz_control, fridge=fridge)
oracle_b0 = evaluate_oracle_fidelity(naive_pulse_rad, dt, H0_true, Hz_control, fridge=fridge)

print(f"Arm A (Twin-Seeded) Initial Oracle F_avg:      {oracle_a0['F_avg']:.4f} (Leakage: {oracle_a0['leakage']:.4f})")
print(f"Arm B (Uncalibrated Guess) Initial Oracle F_avg: {oracle_b0['F_avg']:.4f} (Leakage: {oracle_b0['leakage']:.4f})")

In [ ]:
# Validate that the 9-setting hardware cost minimum aligns with the Oracle infidelity minimum
amp_grid = np.linspace(0.95, 1.1, 25)
est_costs = []
oracle_infids = []

for a_scale in amp_grid:
    test_pulse = build_scaled_pulse(cal_pulse_rad, a_scale, 1.0, dt=dt)
    c_val, _ = measure_cz_hardware_cost(fridge, test_pulse, n_shots=None)
    o_res = evaluate_oracle_fidelity(test_pulse, dt, H0_true, Hz_control, fridge=fridge)
    est_costs.append(c_val)
    oracle_infids.append(1.0 - float(o_res["F_avg"]))

plt.figure(figsize=(7, 4))
plt.plot(amp_grid, oracle_infids, 'o-', color='black', label='True Oracle Infidelity ($1 - F_{avg}$)', markersize=2, linewidth=1)
plt.plot(amp_grid, est_costs, 's--', color='royalblue', label='9-Setting Hardware Cost ($\delta^2/16 + (1-P_{11})/4$)', markersize=2, linewidth=1)
plt.xlabel('Amplitude Scale Factor')
plt.ylabel('Infidelity / Cost')
plt.title('Cost Estimator Validation (Noiseless Sweep on Twin Pulse)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
target_fid = 0.996
n_shots_val = 4000
max_evals = 35
seeds = range(42, 52)  # 10 independent shot-noise seeds per arm

print(f"=== Running Arm A (Twin-Seeded) across {len(seeds)} seeds ===")
arm_a_multi = run_multi_seed_calibration(
    fridge=fridge,
    base_pulse_rad=cal_pulse_rad,
    H0_true=H0_true,
    Hz_control=Hz_control,
    dt=dt,
    init_scale=(1.0, 1.0),
    simplex_step=(0.05, 0.05),
    target_fidelity=target_fid,
    max_iter=max_evals,
    n_shots=n_shots_val,
    prior_experiments=70,  # Accounts for the 70 Swap Chevron settings in NB01
    seeds=seeds,
)

print(f"=== Running Arm B (Uncalibrated Guess) across {len(seeds)} seeds ===")
arm_b_multi = run_multi_seed_calibration(
    fridge=fridge,
    base_pulse_rad=naive_pulse_rad,
    H0_true=H0_true,
    Hz_control=Hz_control,
    dt=dt,
    init_scale=(1.0, 1.0),
    simplex_step=(0.05, 0.05),
    target_fidelity=target_fid,
    max_iter=max_evals,
    n_shots=n_shots_val,
    prior_experiments=0,
    seeds=seeds,
)

print(f"\n--- Multi-Seed Summary (Target F_avg >= {target_fid}) ---")
print(
    f"[Arm A: Twin-Seeded]        Success: {arm_a_multi['success_rate']} | "
    f"Median Experiments to Target: {arm_a_multi['median_experiments_to_target']} | "
    f"Final Median F_avg: {arm_a_multi['median_F_avg'][-1]:.4f}"
)
print(
    f"[Arm B: Uncalibrated Guess] Success: {arm_b_multi['success_rate']} | "
    f"Median Experiments to Target: {arm_b_multi['median_experiments_to_target']} | "
    f"Final Median F_avg: {arm_b_multi['median_F_avg'][-1]:.4f}"
)

In [ ]:
plt.figure(figsize=(9, 5.5))

# Arm A: Median + 25%-75% interquartile shaded band
exp_a = arm_a_multi["exp_grid"]
plt.plot(exp_a, arm_a_multi["median_F_avg"], '-', color='seagreen', linewidth=1,
         label='Arm A: Twin-Seeded (Median Selected $F_{avg}$)')

# Arm B: Median + 25%-75% interquartile shaded band
exp_b = arm_b_multi["exp_grid"]
plt.plot(exp_b, arm_b_multi["median_F_avg"], '-', color='crimson', linewidth=1,
         label='Arm B: Uncalibrated Guess (Median Selected $F_{avg}$)')

plt.axhline(target_fid, color='black', linestyle='--', alpha=0.7, label=f'Target Fidelity ({target_fid})')
plt.axvline(70, color='seagreen', linestyle=':', alpha=0.6, label='NB01 Twin Chevron Budget (70 exps)')

plt.xlabel('Cumulative Physical Hardware Experiments (Settings)')
plt.ylabel('Oracle Average Gate Fidelity of Selected Pulse ($F_{avg}$)')
plt.title(f'Closed-Loop Calibration Race ({len(seeds)} Seeds, {n_shots_val} Shots/Setting)')
plt.ylim(0.97, 1.0)
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()